In [ ]:
"""
Hazard Transition Path Classifier

Core functions to count state transitions in hazard time series and categorize them
into compound hazard evolution pathways. Caller handles data loading, spatial
processing, and temporal aggregation.
"""

import numpy as np
from typing import Dict, Tuple


def count_state_transitions(
    time_series: np.ndarray,
    state_labels: Dict[int, str] = None
) -> Dict[str, int]:
    """
    Count transitions between consecutive hazard states in a time series.
    
    Parameters
    ----------
    time_series : np.ndarray
        1D array of integer hazard states (e.g., [0, 1, 4, 4, 7, ...])
    state_labels : dict, optional
        Mapping from integer states to string labels. Default:
        {0: "No Hazards", 1: "H", 2: "D", 3: "F", 4: "HD", 5: "HF", 6: "DF", 7: "HDF"}
    
    Returns
    -------
    dict
        Transition counts with keys formatted as "STATE1_to_STATE2" for all
        64 possible transitions between 8 states (including self-transitions)
    """
    if state_labels is None:
        state_labels = {
            0: "No Hazards",
            1: "H",
            2: "D",
            3: "F",
            4: "HD",
            5: "HF",
            6: "DF",
            7: "HDF",
        }
    
    # Convert integer states to labels (handle unknown states gracefully)
    labels = np.array([
        state_labels.get(int(state), "No Hazards") 
        for state in time_series
    ])
    
    # Initialize all possible transitions to zero
    all_states = list(state_labels.values())
    transitions = {
        f"{s1}_to_{s2}": 0 
        for s1 in all_states 
        for s2 in all_states
    }
    
    # Count observed transitions
    for i in range(len(labels) - 1):
        key = f"{labels[i]}_to_{labels[i+1]}"
        transitions[key] = transitions.get(key, 0) + 1
    
    return transitions


def categorize_transition_paths(
    transition_counts: Dict[str, int]
) -> Dict[str, int]:
    """
    Categorize state transitions into compound hazard evolution pathways.
    
    Categories follow standard compound hazard progression typology:
      - Single_to_Compound: Isolated hazard → compound event
      - Compound_to_Compound: Compound event → different compound event
      - Recurrent_Compound_Events: Persistence of same compound event
      - Reversal_or_Recovery_Paths: Compound event → simpler state
    
    Parameters
    ----------
    transition_counts : dict
        Output from count_state_transitions()
    
    Returns
    -------
    dict
        Category totals with keys:
          "Single_to_Compound"
          "Compound_to_Compound"
          "Recurrent_Compound_Events"
          "Reversal_or_Recovery_Paths"
    """
    # Define state groups
    single_states = ["H", "D", "F"]
    compound_states = ["HD", "HF", "DF", "HDF"]
    
    # Generate category transition patterns
    single_to_compound = [
        f"{s}_to_{c}" 
        for s in single_states 
        for c in compound_states
    ]
    
    compound_to_compound = [
        f"{c1}_to_{c2}" 
        for c1 in compound_states 
        for c2 in compound_states 
        if c1 != c2  # Exclude self-transitions
    ]
    
    recurrent_compound = [
        f"{c}_to_{c}" 
        for c in compound_states
    ]
    
    reversal_paths = (
        # Compound → single hazard
        [f"{c}_to_{s}" for c in compound_states for s in single_states] +
        # HDF-specific degradation paths (HDF → partial compound)
        ["HDF_to_HD", "HDF_to_HF", "HDF_to_DF"]
    )
    
    # Aggregate counts per category
    return {
        "Single_to_Compound": sum(
            transition_counts.get(t, 0) for t in single_to_compound
        ),
        "Compound_to_Compound": sum(
            transition_counts.get(t, 0) for t in compound_to_compound
        ),
        "Recurrent_Compound_Events": sum(
            transition_counts.get(t, 0) for t in recurrent_compound
        ),
        "Reversal_or_Recovery_Paths": sum(
            transition_counts.get(t, 0) for t in reversal_paths
        )
    }


def analyze_hazard_transitions(
    time_series: np.ndarray,
    state_labels: Dict[int, str] = None
) -> Tuple[Dict[str, int], Dict[str, int]]:
    """
    End-to-end analysis of hazard transition pathways for a single time series.
    
    Parameters
    ----------
    time_series : np.ndarray
        1D array of integer hazard states
    state_labels : dict, optional
        State mapping (see count_state_transitions)
    
    Returns
    -------
    tuple
        (transition_counts, category_totals)
        - transition_counts: Raw transition frequencies (64-state matrix)
        - category_totals: Aggregated counts by pathway category
    """
    transitions = count_state_transitions(time_series, state_labels)
    categories = categorize_transition_paths(transitions)
    return transitions, categories